# Prepare MD files: combine EDR → combine XTC → process XTC

Edit the system folder in **1. Settings**, then **Run All**. This notebook uses the
tested Bash scripts: it detects available production parts, combines EDR and XTC
in the raw folder, centres on **Protein only**, retains **System**, and writes the
full processed trajectory and every-100th-frame VMD preview in `analysis/`.

Only Python's standard library, Bash and GROMACS are required. Select a notebook
kernel whose environment can find `gmx`, or set its full executable path below.
The production TPR must match the XTC atom order. Parts must come from the same
production run with continuous timestamps; filenames alone cannot verify this.

For a new system, the six SH/TXT files are installed in its `analysis/` folder.
Existing local scripts and settings are preserved. Existing non-empty outputs
are reused if no source file is newer; this is **not** an integrity check.
Change output filenames when adding new continuation parts. Interrupted stages
are marked and will not be silently reused. No production MD is launched.


## 1. Settings

For another system, change `SIMULATION_DIR`. Change `PARTS_PREFIX` only if its
continuation filenames differ (for example `step8_production_2us`). `OUTPUT_STEM`
is a filename label, not a time limit. Set `DRY_RUN=True` to inspect the plan
without writing files or running GROMACS.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
from datetime import datetime

SIMULATION_DIR = Path("/Users/k20098771/Data/MD_projects/PHA/MD_data/gromacs/PHA_new_Enzyme/ANC55_P3HO4_gromacs")
FIRST_PREFIX = "step7_production"
PARTS_PREFIX = "step8_production_1us"
TPR_FILENAME = "step7_production.tpr"
OUTPUT_STEM = "production_combined_1us"
PROCESSED_FILENAME = "processed_protein_centered.xtc"
PREVIEW_FILENAME = "processed_protein_centered_every100.xtc"
STRIDE = 100
GMX = "gmx"  # Or an absolute executable path, e.g. /opt/homebrew/bin/gmx
DRY_RUN = False

# Works when VS Code starts the kernel in this folder or an ancestor in the repo.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src/md_simulation_scripts/trajectory_preparation/templates").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start the kernel inside the iPHASimulator repository, or set REPO_ROOT to its absolute path.")
TEMPLATES = REPO_ROOT / "src/md_simulation_scripts/trajectory_preparation/templates"


## 2. Check inputs and prepare the plan

The detected production files are printed below. Equilibration files, existing
combined files and unrelated continuation prefixes are excluded.


In [ ]:
def require_file(path):
    if not path.is_file() or not os.access(path, os.R_OK) or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing, empty or unreadable file: {path}")

def read_settings(path):
    result = {}
    for line in path.read_text().splitlines():
        if not line.strip() or line.lstrip().startswith("#"):
            continue
        key, separator, value = line.partition("=")
        if not separator or not value or key in result:
            raise ValueError(f"Invalid or duplicate setting in {path}: {line}")
        result[key] = value
    return result

raw = SIMULATION_DIR.expanduser().resolve()
if not raw.is_dir():
    raise NotADirectoryError(raw)
if not isinstance(STRIDE, int) or isinstance(STRIDE, bool) or STRIDE < 1:
    raise ValueError("STRIDE must be a positive integer.")
for name in (FIRST_PREFIX, PARTS_PREFIX, TPR_FILENAME, OUTPUT_STEM,
             PROCESSED_FILENAME, PREVIEW_FILENAME):
    if not name or name in (".", "..") or Path(name).name != name or any(c in name for c in "*?[]\r\n"):
        raise ValueError(f"Use a plain filename/prefix, without paths or wildcards: {name!r}")
if not TPR_FILENAME.endswith(".tpr") or not all(n.endswith(".xtc") for n in (PROCESSED_FILENAME, PREVIEW_FILENAME)):
    raise ValueError("Use a .tpr topology and .xtc processed/preview filenames.")
if PROCESSED_FILENAME == PREVIEW_FILENAME:
    raise ValueError("Processed and preview filenames must differ.")
if shutil.which(GMX) is None:
    raise FileNotFoundError(f"Cannot find {GMX}; set GMX to the GROMACS executable path.")

analysis = raw / "analysis"
combined_edr = raw / f"{OUTPUT_STEM}.edr"
combined_xtc = raw / f"{OUTPUT_STEM}.xtc"
processed = analysis / PROCESSED_FILENAME
preview = analysis / PREVIEW_FILENAME
tpr = raw / TPR_FILENAME
require_file(tpr)

input_files = {}
for suffix in ("edr", "xtc"):
    parts = sorted(raw.glob(f"{PARTS_PREFIX}.part[0-9][0-9][0-9][0-9].{suffix}"))
    if not parts:
        raise FileNotFoundError(f"No numbered {suffix.upper()} parts found for {PARTS_PREFIX}.")
    inputs = [raw / f"{FIRST_PREFIX}.{suffix}", *parts]
    for path in inputs:
        require_file(path)
    input_files[suffix] = inputs
    print(f"{suffix.upper()}: {len(inputs)} production files")
    for path in inputs:
        print("  ", path.name)

desired_settings = {
    "combine_edr_instruction.txt": {"GMX": GMX, "OUTPUT": f"../{combined_edr.name}",
        "FIRST": f"../{FIRST_PREFIX}.edr", "PARTS_PREFIX": f"../{PARTS_PREFIX}"},
    "combine_xtc_instruction.txt": {"GMX": GMX, "OUTPUT": f"../{combined_xtc.name}",
        "FIRST": f"../{FIRST_PREFIX}.xtc", "PARTS_PREFIX": f"../{PARTS_PREFIX}"},
    "instrcution.txt": {"INPUT": f"../{combined_xtc.name}", "TPR": f"../{TPR_FILENAME}",
        "INDEX": "protein_center.ndx", "CENTER_GROUP": "Protein", "OUTPUT_GROUP": "System",
        "STRIDE": str(STRIDE), "PROCESSED": PROCESSED_FILENAME, "PREVIEW": PREVIEW_FILENAME, "GMX": GMX},
}
for name, settings in desired_settings.items():
    path = analysis / name
    if path.is_symlink():
        raise ValueError(f"Settings must not be symbolic links: {path}")
    if path.exists() and read_settings(path) != settings:
        raise ValueError(f"Notebook settings differ from {path}. Align the settings in the notebook and TXT before running; existing settings are preserved.")

steps = [
    ("combine_edr.sh", input_files["edr"], [combined_edr]),
    ("combine_xtc.sh", input_files["xtc"], [combined_xtc]),
    ("process_trajectory.sh", [combined_xtc, tpr], [processed, preview]),
]
statuses = {}
for script, sources, outputs in steps:
    marker = analysis / f".{script}.incomplete"
    if marker.exists():
        raise RuntimeError(f"An interrupted/failed stage is marked by {marker}. Inspect its outputs and log before removing the marker and retrying.")
    exists = [path.exists() or path.is_symlink() for path in outputs]
    if any(exists):
        if not all(exists):
            raise RuntimeError(f"Only some outputs exist for {script}. Inspect them and choose fresh output filenames.")
        for path in outputs:
            if path.is_symlink():
                raise ValueError(f"Output must not be a symbolic link: {path}")
            require_file(path)
        if any(not source.is_file() for source in sources):
            raise RuntimeError(f"Existing outputs for {script} have missing source files.")
        if max(source.stat().st_mtime_ns for source in sources) > min(output.stat().st_mtime_ns for output in outputs):
            raise RuntimeError(f"Newer inputs found for {script}. Choose fresh output filenames rather than reusing stale files.")
        statuses[script] = "reuse existing outputs (not revalidated)"
    else:
        statuses[script] = "run"
    print(f"{script}: {statuses[script]}")

# Install only missing files; keep the local SH/TXT workflow available afterwards.
if not DRY_RUN:
    analysis.mkdir(exist_ok=True)
    for script, _, _ in steps:
        destination = analysis / script
        if destination.is_symlink():
            raise ValueError(f"Script must not be a symbolic link: {destination}")
        if not destination.exists():
            shutil.copyfile(TEMPLATES / script, destination)
    for name, settings in desired_settings.items():
        destination = analysis / name
        if not destination.exists():
            comments = "\n".join(line for line in (TEMPLATES / name).read_text().splitlines()
                                  if not line.strip() or line.lstrip().startswith("#"))
            with destination.open("x") as handle:
                handle.write(comments.rstrip() + "\n\n")
                handle.write("".join(f"{key}={value}\n" for key, value in settings.items()))

def run_step(script):
    sources, outputs = next((s, o) for name, s, o in steps if name == script)
    if statuses[script] != "run":
        print(f"{script}: {statuses[script]}")
        return
    if DRY_RUN:
        print(f"Would run bash {script} in {analysis}")
        return
    for source in sources:
        require_file(source)
    log_dir = analysis / "preparation_logs"
    log_dir.mkdir(exist_ok=True)
    log = log_dir / f"{datetime.now():%Y%m%d_%H%M%S_%f}_{script}.log"
    marker = analysis / f".{script}.incomplete"
    with marker.open("x") as handle:
        handle.write(f"Log: {log}\nOutputs: {', '.join(str(p) for p in outputs)}\n")
    print(f"Running {script}. GROMACS output is saved to {log}", flush=True)
    with log.open("w") as handle:
        result = subprocess.run(["bash", str(analysis / script)], cwd=analysis,
                                stdout=handle, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f"{script} failed (exit {result.returncode}). Inspect {log}. Later stages were not run.")
    for output in outputs:
        require_file(output)
    marker.unlink()
    statuses[script] = "completed"
    print(f"Completed {script}")


## 3. Combine EDR

Write the combined energy file in the raw simulation folder. This combines
energy records; energy extraction and plotting are performed in the analysis notebooks.


In [ ]:
run_step("combine_edr.sh")


## 4. Combine XTC

Write the combined trajectory in the raw simulation folder, preserving original
timestamps and using GROMACS's default handling of overlapping frames.


In [ ]:
run_step("combine_xtc.sh")


## 5. Centre the protein and create the VMD preview

Apply `-pbc mol -ur compact -center`, selecting **Protein** for centring and
**System** for output. Save all processed frames, then every `STRIDE`th frame
for VMD. The separate `protein_center.ndx` is generated from the production TPR
and remains specific to this simulation. No fitting or `nojump` is applied.


In [ ]:
run_step("process_trajectory.sh")


## 6. Output files

Use the full processed trajectory for analysis. Enzyme–PHA distances and contacts
must still use each frame's periodic box. For VMD, load the matching raw-folder
`step6.2_npt.gro`, then add the preview XTC to the same molecule.

After a successful run, the local SH/TXT files can also be used directly from
`analysis/` without opening this notebook.


In [ ]:
print("Plan only" if DRY_RUN else "Preparation finished")
for path in (combined_edr, combined_xtc, processed, preview):
    size = f"{path.stat().st_size / 1024**3:.3f} GiB" if path.is_file() else "not created"
    print(f"{path}: {size}")
